# Read counts after a pasilla knockdown

This notebook holds seven sample names and the raw RNA-seq read counts of six genes, and
computes mean counts with plain Python: which genes have a mean above 1000 reads, the
mean per condition, and which genes have fewer reads after the knockdown. Run it from the
top: the first cell holds the data.

In [ ]:
# RNA-seq read counts of six genes in seven samples of Drosophila melanogaster
# Source: the Bioconductor data package pasilla (see the experiment's README)

# The sample names, in the same order as the counts below
samples = ["ctrl_rep1", "ctrl_rep2", "ctrl_rep3", "ctrl_rep4", "kd_rep1", "kd_rep2", "kd_rep3"]

# Read counts: one list per gene (FlyBase gene ID), one number per sample
counts = {
    "FBgn0000008": [92, 161, 76, 70, 140, 88, 70],
    "FBgn0000014": [5, 1, 0, 0, 4, 0, 0],
    "FBgn0000042": [101664, 120163, 45880, 53201, 127363, 76099, 83164],
    "FBgn0025111": [451, 653, 279, 306, 4186, 2381, 2686],
    "FBgn0039155": [1527, 2220, 741, 931, 70, 38, 52],
    "FBgn0261552": [7022, 16750, 5224, 4760, 4233, 1335, 1394],
}

## How Python code looks

The LLM's code will use comments, variables, data types, lists, dictionaries, loops and
conditions. Run this cell and match each comment with the line below it.

In [ ]:
# Variables and data types
gene = "FBgn0000008"            # a text (str)
threshold = 1000                # a whole number (int)
first_count = counts[gene][0]   # look up the gene in the dictionary, then the first number in its list
print(gene, type(gene), threshold, type(threshold), first_count)

# A loop over the dictionary, with a condition inside it
for gene_id, values in counts.items():
    total = sum(values)
    if total < 100:
        print(gene_id, "has few reads:", total)
    else:
        print(gene_id, "has", total, "reads in", len(values), "samples")

## Small exercise

In [ ]:
# Small exercise: the genes whose mean count over all samples is above the threshold
threshold = 1000

for gene_id, values in counts.items():
    mean = sum(values) / len(values)   # mean over all seven samples
    if mean > threshold:
        print(f"{gene_id}: mean {mean:.1f}")

# Check by hand: one gene above and one below the threshold
print(sum(counts["FBgn0025111"]) / 7)   # 1563.1, listed
print(sum(counts["FBgn0039155"]) / 7)   # 797.0, not listed

## Larger exercise

An LLM was given the first cell of this notebook and this prompt:

> Sample names look like `ctrl_rep1`: a condition, an underscore, `rep` and a replicate
> number. Write plain Python that turns each name in `samples` into a dictionary with the
> condition and the replicate number, then computes the mean count per condition for each
> gene in `counts`.

Its answer is pasted unchanged in the next cell.

In [ ]:
def parse_sample_name(name):
    """Split a sample name like 'ctrl_rep1' into its condition and replicate number."""
    condition, replicate = name.split("_")
    # Fixed: the replicate part is "rep1", not "1"; remove "rep" before converting
    return {"condition": condition, "replicate": int(replicate.removeprefix("rep"))}


# Parse all sample names
sample_info = {}
for name in samples:
    sample_info[name] = parse_sample_name(name)

# Compute the mean count per condition for each gene
mean_counts = {}
for gene_id, values in counts.items():
    per_condition = {}
    for name, value in zip(samples, values):
        condition = sample_info[name]["condition"]
        if condition not in per_condition:
            per_condition[condition] = []
        per_condition[condition].append(value)

    mean_counts[gene_id] = {}
    for condition, condition_values in per_condition.items():
        mean_counts[gene_id][condition] = sum(condition_values) / len(condition_values)

for gene_id, means in mean_counts.items():
    print(gene_id, means)

What was wrong: the LLM's code raised `ValueError: invalid literal for int() with base 10:
'rep1'` in `parse_sample_name`. Splitting `ctrl_rep1` at the underscore gives `rep1`, not
`1`, so `int()` cannot convert it. We found it by reading the traceback from the bottom and
printing the result of `split`, and fixed it by removing `rep` before converting.

In [ ]:
# Debugging step by step: what does int() get?
print("ctrl_rep1".split("_"))   # ['ctrl', 'rep1']: the replicate part is 'rep1'

In [ ]:
# Check one mean by hand: kd mean of FBgn0039155
print(counts["FBgn0039155"][4:], (70 + 38 + 52) / 3)

# Genes with a lower mean in the knockdown than in the control
for gene_id, means in mean_counts.items():
    if means["kd"] < means["ctrl"]:
        print(gene_id, round(means["ctrl"], 1), "->", round(means["kd"], 1))

## Bonus exercise

In [ ]:
# Made-up sample names to test the parser. The rule: <condition>_rep<number>,
# where the condition is ctrl or kd.
more_samples = ["ctrl_rep5", "kd-rep4", "kd_rep", "KD_rep5", "kd_rep4", "ctrl_rep6_new"]

In [ ]:
CONDITIONS = ["ctrl", "kd"]


def parse_sample_name(name):
    """Split a sample name like 'ctrl_rep1' into condition and replicate number.

    Return None and print the reason if the name does not follow <condition>_rep<number>.
    """
    parts = name.split("_")
    if len(parts) != 2:
        print(f"Malformed: {name!r} must have exactly one underscore")
        return None
    condition, replicate = parts
    if condition not in CONDITIONS:
        print(f"Malformed: {name!r} has unknown condition {condition!r}")
        return None
    number = replicate.removeprefix("rep")
    if not replicate.startswith("rep") or not number.isdigit():
        print(f"Malformed: {name!r} needs 'rep' and a number after the underscore")
        return None
    return {"condition": condition, "replicate": int(number)}


parsed = {}
for name in samples + more_samples:
    info = parse_sample_name(name)
    if info is not None:
        parsed[name] = info

print(len(parsed), "names parsed:", list(parsed))